# AIPerf Use Case 5 — Time-Sliced Analysis: Performance Over Time

A single summary table averages over the whole run and hides *when* things were slow. `--slice-duration N` buckets results into N-second windows so you can see warm-up, degradation, and load-spike behavior — e.g. a cold-start first slice with a noticeably higher TTFT than the steady-state slices that follow.

These use-case notebooks demonstrate AIPerf capabilities directly.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements](#2-requirements)
3. [Input — workload and slice duration](#3-input--workload-and-slice-duration)
4. [Test run](#4-test-run)
5. [Results analysis](#5-results-analysis)

## 1. Overview

With `--slice-duration`, AIPerf additionally writes `profile_export_aiperf_timeslices.csv` / `.json`: per-window request counts and metric statistics. What it surfaces:

- **Warm-up / cold start** — first slice slower despite fewer requests (model/kernel warm-up, cache fills).
- **Degradation over time** — creeping TTFT/ITL from memory leaks, KV-cache fragmentation, GC pauses.
- **Load-pattern response** — when replaying a day-long trace with spikes, how latency tracked each spike.

This is directly relevant to the repo's **Sustained/Soak scenario** (`docs/scenarios/sustained-soak-load.md`, `model-selection/sustained-soak-load/`): the soak script varies wall-clock duration to catch exactly these effects, and `--slice-duration` is the built-in way to see them *within* a run instead of only in the end-of-run aggregate.

## 2. Requirements

- `aiperf` CLI installed and on `PATH` (from the NGC AIPerf image, or `pip install aiperf`). Pin whatever version you use (repo convention: record the AIPerf version per run) — this notebook was written against `release/0.3.0`.
- A reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...) serving the model under test.
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).
- Tip: AIPerf's live dashboard is designed for a terminal. If it renders poorly inside Jupyter, check `aiperf profile --help` for a plain/simple UI option in your version.


In [ ]:
import json
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

aiperf_path = shutil.which("aiperf")
if aiperf_path is None:
    print("aiperf CLI not found on PATH — install it before running the Test run section.")
else:
    print(f"aiperf found at: {aiperf_path}")
    version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
    print((version.stdout or version.stderr).strip())


## 3. Input — workload and slice duration

Any workload works. Choose `--slice-duration` relative to run length — enough slices to see a trend, enough requests per slice for stable statistics:

| Run length | Suggested slice |
|---|---|
| ~1 min demo | 10 s |
| 20–60 min soak | 1–5 min |
| multi-hour soak | 10–15 min |

A trace-based workload (e.g. the Mooncake 5-min/5× slice from the Use Case 3 notebook) works too; this notebook defaults to synthetic traffic so it runs standalone.

## 4. Test run

In [ ]:
def run_aiperf(cmd):
    """Print and run an aiperf command from the repo root, streaming output into the notebook."""
    print("$ " + " \\\n    ".join(cmd))
    result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
    print(f"\nexit code: {result.returncode}")
    return result

# ---- Required ----------------------------------------------------------
URL = ""
MODEL = ""       # leave empty to auto-detect from {URL}/v1/models; set to override

SLICE_DURATION = 10        # seconds per bucket
INPUT_FILE = ""            # optional mooncake-format JSONL (e.g. the UC3 slice); empty = synthetic
CONCURRENCY = 10
REQUEST_COUNT = 300        # enough requests to fill several slices
TOKENIZER = ""
# Relative to REPO_ROOT (aiperf runs with cwd=REPO_ROOT) — keep artifacts under notebooks/.
OUTPUT_DIR = "notebooks/artifacts/aiperf-uc5-timeslices"

assert URL, "Set URL above."

# Ask the endpoint what it serves — verifies the URL is reachable and spares
# the user a second copy-paste. Multi-model servers: first model wins.
if not MODEL:
    models_url = URL.rstrip("/") + "/v1/models"
    with urllib.request.urlopen(models_url, timeout=10) as resp:
        served = json.load(resp)["data"]
    assert served, f"{models_url} is reachable but lists no models."
    if len(served) > 1:
        print(f"{models_url} lists {len(served)} models: {[m['id'] for m in served]}")
    MODEL = served[0]["id"]
print(f"URL   : {URL}")
print(f"MODEL : {MODEL}")

cmd = [
    "aiperf", "profile",
    "--model", MODEL, "--url", URL,
    "--endpoint-type", "chat", "--streaming",
    "--tokenizer", TOKENIZER or MODEL,
    "--artifact-dir", OUTPUT_DIR,
    "--slice-duration", str(SLICE_DURATION),
]
if INPUT_FILE:
    cmd += ["--input-file", INPUT_FILE, "--custom-dataset-type", "mooncake_trace", "--fixed-schedule"]
else:
    cmd += ["--concurrency", str(CONCURRENCY), "--request-count", str(REQUEST_COUNT),
            # NOTE: --synthetic-input-tokens-mean / --output-tokens-mean are the
            # canonical long names AIPerf uses for ISL/OSL. Used here because they
            # are guaranteed across AIPerf versions and match this repo's scenario
            # scripts (e.g. --output-tokens-mean in model-selection/scripts/run_content_generation.sh).
            "--synthetic-input-tokens-mean", "1000", "--output-tokens-mean", "500"]
run_aiperf(cmd)


## 5. Results analysis

The timeslice CSV is in **long format**: one row per `(Timeslice, Metric, Stat)` combination, with columns `Timeslice, Metric, Unit, Stat, Value`. Per-request metrics (TTFT, ITL, request latency, ...) get the full stat set (avg/min/max/percentiles/std) per slice; run-level metrics (Request Count, Output Token Throughput, Benchmark Duration) appear as a single `avg` row per slice. The helper below extracts one metric+stat as a series over slices.

In [ ]:
import pandas as pd

artifact_dir = REPO_ROOT / OUTPUT_DIR
slices_csv = next(artifact_dir.rglob("*timeslices*.csv"), None)
assert slices_csv is not None, f"No timeslices CSV under {artifact_dir} — was --slice-duration set?"

slices = pd.read_csv(slices_csv)  # long format: Timeslice, Metric, Unit, Stat, Value
n_slices = slices["Timeslice"].nunique()
print(f"{n_slices} slices from {slices_csv.name}")
print(f"Metrics available: {sorted(slices['Metric'].unique())}")


def slice_series(metric, stat="avg"):
    """One metric+stat as a Series indexed by slice number."""
    m = slices[(slices["Metric"] == metric) & (slices["Stat"] == stat)]
    return m.set_index("Timeslice")["Value"].sort_index()


# Raw rows for slice 0 only, in the CSV's native long format (Timeslice,
# Metric, Unit, Stat, Value) — shows the structure slice_series() reads from.
slices[slices["Timeslice"] == 0].head(20)


In [ ]:
import matplotlib.pyplot as plt

ttft = slice_series("Time to First Token")          # ms, avg per slice
tput = slice_series("Output Token Throughput")      # tokens/sec, system-wide per slice
count = slice_series("Request Count")               # requests completed per slice

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, series, title in [(axes[0], ttft, "TTFT avg per slice (ms)"),
                          (axes[1], tput, "Output token throughput per slice (tokens/sec)"),
                          (axes[2], count, "Requests per slice")]:
    if series.empty:
        ax.set_title(f"{title}: metric not found — check the metric list above")
        continue
    ax.plot(series.index, series.values, marker="o")
    ax.set_title(title)
    ax.set_xlabel(f"slice # ({SLICE_DURATION}s each)")
plt.tight_layout()


In [ ]:
# Simple trend checks: cold start (first slice vs. steady state) and drift (late vs. mid run).
# The last slice is often a partial window with few requests — drop it from the drift check.
if len(ttft) >= 3:
    first = ttft.iloc[0]
    steady = ttft.iloc[1:].median()
    print(f"Slice 0 TTFT: {first:.0f} ms vs. steady-state median {steady:.0f} ms "
          f"({100 * (first - steady) / steady:+.0f}%)  -> cold start if strongly positive")

    body = ttft.iloc[1:-1] if len(ttft) > 3 else ttft.iloc[1:]
    half = len(body) // 2
    early, late = body.iloc[:half].median(), body.iloc[half:].median()
    print(f"Early-half TTFT median: {early:.0f} ms, late-half: {late:.0f} ms "
          f"({100 * (late - early) / early:+.0f}%)  -> degradation if strongly positive")
else:
    print("Fewer than 3 slices — run longer (or shorten SLICE_DURATION) for trend checks.")


### Notes

- A cold-start slice with fewer requests than surrounding slices but a *higher* average latency is the signature of warm-up rather than load. If you benchmark for comparisons (Model Selection / Sizing), that's why the repo's scripts send warm-up requests first (`--warmup-request-count`).
- For soak runs, sustained upward TTFT/ITL drift across slices (with constant load) is the leak/fragmentation signal the Sustained/Soak scenario exists to catch — consider adding `--slice-duration` there when it's next revised.
- The per-request JSONL (Use Case 2 notebook) can reproduce any custom bucketing if the fixed slices don't fit your analysis.